## Notebook to start a spark instance in an ArcGIS Pro Session.

In [ ]:
import os
from spark_esri import spark_start, spark_stop

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
import arcpy
import geofunctions as S

In [ ]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    # Adjust to your environment.
    "spark.jars": os.path.expanduser("~/geofunctions-0.9.jar"),
}
spark = spark_start(config=config)

In [ ]:
spark.version

### Sample usage. Create random points in the current map extent and place them on the Map.

In [ ]:
project = arcpy.mp.ArcGISProject("CURRENT")

extent = project.activeView.camera.getExtent()
xmin = extent.XMin
ymin = extent.YMin
dx = extent.XMax - extent.XMin
dy = extent.YMax - extent.YMin

In [ ]:
(spark
    .range(1000)
    # Get Random x/y value.
    .withColumn("x", F.rand() * dx + xmin)
    .withColumn("y", F.rand() * dy + ymin)
    # Create a point geometry
    .withColumn("geom", S.st_point())
    # Convert the spark dataframe to a feature class.
    .to_feature_class(
        "MyPoints",
        sp_ref = -1,  # Use current map SR.
)
)